# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset, which documents a mass spectrometry analysis of extracellular vesicles from human mast cells, using the `mlcroissant` library and Python. All references to data entities (record sets, fields) use their Croissant `@id` values to ensure clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant
# Install plotting libraries as well
!pip install matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset name: ", metadata.name)
print("Description: ", metadata.description)

## 2. Data Overview

Review available record sets, their fields, and the Croissant `@id` values. This ensures all downstream usage references the correct schema identifiers.


In [ ]:
# List available record sets with their @id and human-readable name
record_sets = []
if hasattr(metadata, "record_sets"):
    # mlcroissant >=0.5.0; record_sets attribute
    rs_list = metadata.record_sets
else:
    rs_list = getattr(metadata, "recordSet", getattr(metadata, "record_sets", []))
if not rs_list:
    # fallback: try reading from metadata dict, if possible
    try:
        rs_list = metadata.to_json().get("recordSet", [])
    except Exception:
        rs_list = []
# Normalize record_sets into objects
for rs in rs_list:
    try:
        # rs is object-like in mlcroissant>=0.8.0
        record_sets.append((rs.id, rs.name if hasattr(rs, "name") else ""))
    except Exception:
        # rs is a dict with '@id' and 'name'
        record_sets.append((rs.get("@id", str(rs)), rs.get("name", "")))

if not record_sets:
    print("No record sets found in the metadata.")
else:
    print("Available Record Sets:")
    for rid, name in record_sets:
        print(f"  @id: {rid}\tname: {name}")

# Examine fields for each record set
record_set_fields = {}
for rid, _ in record_sets:
    print(f"\nFields for RecordSet {rid}:")
    try:
        rs_obj = next(rs for rs in rs_list if getattr(rs, 'id', rs.get('@id', '')) == rid)
    except Exception:
        continue
    # Fields may be object or dict, depending on loader
    field_list = getattr(rs_obj, "fields", getattr(rs_obj, "field", getattr(rs_obj, "fields", [])))
    record_set_fields[rid] = []
    for f in field_list:
        # f is object or dict
        field_id = getattr(f, "id", f.get("@id", str(f)))
        field_name = getattr(f, "name", f.get("name", ""))
        record_set_fields[rid].append(field_id)
        print(f"  @id: {field_id}\tname: {field_name}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis, referencing all entities using their `@id`. Adjust the code below to select the specific record set and fields you wish to work with.

In [ ]:
# Specify the record set(s) to analyze
# For this dataset, there is typically a main record set. Use @ids discovered above.
if record_sets:
    # Use all available record sets
    selected_record_set_ids = [rid for rid, _ in record_sets]
else:
    selected_record_set_ids = []
dataframes = {}

for record_set_id in selected_record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet {record_set_id}")
    except Exception as e:
        print(f"Error loading records for {record_set_id}:", e)
if dataframes:
    # Show columns from the first loaded dataframe
    first_rs_id = selected_record_set_ids[0]
    print("\nColumns in first record set:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print("No dataframes loaded for available record sets.")

## 4. Exploratory Data Analysis (EDA)

We can filter, normalize, and group data from a numeric field. Please refer to the printed field list above to choose appropriate fields. (For demonstration, field names are directly referenced as their `@id`.)

In [ ]:
# Example configuration:
# Replace these @id's with those matching your chosen columns (see above outputs)
record_set_id = first_rs_id if dataframes else None
# Guess the likely numeric field @id (common in proteomics: 'MW' for molecular weight, 'sequence_coverage', etc.)
if record_set_id and dataframes[record_set_id].shape[1] > 0:
    # Try to auto-detect a numeric field
    df = dataframes[record_set_id]
    sample_numeric_id = None
    for col in df.columns:
        if df[col].dtype in [float, int] or pd.api.types.is_numeric_dtype(df[col]):
            sample_numeric_id = col
            break
    if not sample_numeric_id:
        # fallback: try to select one by name
        for try_id in ["MW", "sequence_coverage", "normalized_abundance", "score"]:
            for col in df.columns:
                if try_id.lower() in col.lower():
                    sample_numeric_id = col
                    break
            if sample_numeric_id:
                break
    numeric_field_id = sample_numeric_id if sample_numeric_id else df.columns[0]
    
    # Now perform filtering and normalization
    threshold = df[numeric_field_id].quantile(0.5) # median
    print(f"Filtering records where {numeric_field_id} > {threshold:.2f}")
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Attempt to group by a likely categorical field
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and df[col].nunique() < min(8, len(df) // 10):
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id} (mean of numeric fields):")
        print(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")
else:
    print("No records or columns to analyze for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. Adapt field `@id` values as appropriate from the earlier listing.


In [ ]:
if record_set_id and dataframes:
    df = dataframes[record_set_id]
    if df.shape[1] > 1:
        # Plot histogram of the main numeric field
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.show()

        # Scatterplot if sufficient columns
        candidate_x = numeric_field_id
        candidate_y = None
        for col in df.columns:
            if col != candidate_x and pd.api.types.is_numeric_dtype(df[col]):
                candidate_y = col
                break
        if candidate_y:
            plt.figure(figsize=(7,5))
            sns.scatterplot(data=df, x=candidate_x, y=candidate_y)
            plt.title(f"{candidate_x} vs {candidate_y}")
            plt.show()
    else:
        print("Not enough numeric columns for visualization.")
else:
    print("No data to visualize.")

## 6. Conclusion

This notebook demonstrated how to load, inspect, and perform basic exploration and visualization of a Croissant-standardized dataset (FAIR²) via the `mlcroissant` library. All entities—record sets, fields, and columns—were referenced by their Croissant `@id` for interoperability. 

You can further extend these analyses by selecting more specific fields, joining to external datasets (where compatible), and leveraging detailed schema documentation provided in the Croissant JSON-LD for advanced data science workflows.
